# Analyse Exploratoire des Données (EDA) — Customer Churn

**Projet M1 Data Science — Mastère Dev. Manager Full Stack, 2025-26**  
**Auteurs : VERSAYO Franklin & Danny Navarro Cordeau**  
**Professeure : Sarah Malaeb**

---

## Objectifs

1. Auditer la qualité des données (valeurs manquantes, outliers, déséquilibre)
2. Analyser la distribution des variables numériques et catégorielles
3. Identifier les features les plus discriminantes pour le churn
4. Visualiser les corrélations et les patterns comportementaux
5. Formuler des hypothèses pour la modélisation

---

**Dataset** : `data/raw/customer_churn.csv`  
**Cible** : `churn` (0 = non-churn, 1 = churn)

In [ ]:
import sys
from pathlib import Path

# Ajout de la racine du projet
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='Set2')

from src.config import (
    RAW_CSV, NUMERIC_FEATURES, CATEGORICAL_FEATURES,
    TARGET_CHURN, TARGET_REVENUE, FIGURES_DIR
)

print('Librairies chargées ✓')

## 1. Chargement et aperçu général

In [ ]:
df = pd.read_csv(RAW_CSV)
print(f'Shape : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## 2. Audit qualité des données

In [ ]:
# Valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.concat([missing, missing_pct], axis=1, keys=['count', '%'])
missing_df = missing_df[missing_df['count'] > 0].sort_values('%', ascending=False)

if len(missing_df) > 0:
    print('Colonnes avec valeurs manquantes :')
    display(missing_df)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_df['%'].plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('% valeurs manquantes')
    ax.set_title('Taux de valeurs manquantes par colonne', fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'eda_missing_values.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('✓ Aucune valeur manquante détectée.')

In [ ]:
# Déséquilibre de classes
churn_counts = df[TARGET_CHURN].value_counts()
churn_pct    = df[TARGET_CHURN].value_counts(normalize=True) * 100

print('Distribution de la variable cible (churn) :')
print(pd.concat([churn_counts, churn_pct.round(1)], axis=1, keys=['count', '%']))
print(f'\nTaux de churn : {df[TARGET_CHURN].mean():.2%}')
print(f'Ratio de déséquilibre : 1:{churn_counts[0]/churn_counts[1]:.1f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Pie
axes[0].pie(churn_counts, labels=['Non-Churn', 'Churn'],
            autopct='%1.1f%%', colors=['#2ca02c', '#d62728'],
            startangle=90, explode=[0, 0.08])
axes[0].set_title('Répartition du churn', fontweight='bold')

# Bar
axes[1].bar(['Non-Churn (0)', 'Churn (1)'], churn_counts.values,
            color=['#2ca02c', '#d62728'], edgecolor='black')
for i, v in enumerate(churn_counts.values):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')
axes[1].set_ylabel('Nombre de clients')
axes[1].set_title('Déséquilibre des classes', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_class_imbalance.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Distribution des variables numériques

In [ ]:
num_cols = [c for c in NUMERIC_FEATURES if c in df.columns]
n_cols = 3
n_rows = (len(num_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    df[df[TARGET_CHURN] == 0][col].hist(
        ax=axes[i], bins=30, alpha=0.6, color='#2ca02c', label='Non-Churn', density=True
    )
    df[df[TARGET_CHURN] == 1][col].hist(
        ax=axes[i], bins=30, alpha=0.6, color='#d62728', label='Churn', density=True
    )
    axes[i].set_title(col, fontweight='bold', fontsize=10)
    axes[i].legend(fontsize=8)
    axes[i].grid(alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution des variables numériques — Churn vs Non-Churn',
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_numeric_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Variables catégorielles — Taux de churn

In [ ]:
cat_cols = [c for c in CATEGORICAL_FEATURES if c in df.columns]
n_cols = 3
n_rows = (len(cat_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

mean_churn = df[TARGET_CHURN].mean()

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)[TARGET_CHURN].mean().sort_values(ascending=True)
    colors = ['#d62728' if v >= mean_churn else '#2ca02c' for v in churn_rate.values]
    churn_rate.plot(kind='barh', ax=axes[i], color=colors)
    axes[i].axvline(mean_churn, color='navy', linestyle='--', linewidth=1.5, label=f'Moy. {mean_churn:.1%}')
    axes[i].set_title(f'Churn rate — {col}', fontweight='bold', fontsize=10)
    axes[i].set_xlabel('Taux de churn')
    axes[i].legend(fontsize=8)
    axes[i].grid(axis='x', alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Taux de churn par variable catégorielle',
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_categorical_churn_rates.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Matrice de corrélation

In [ ]:
corr_cols = num_cols + [TARGET_CHURN]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, ax=ax, mask=mask,
    annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.3,
    annot_kws={'size': 8}
)
ax.set_title('Matrice de corrélation — Features numériques + Churn', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_correlation_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Corrélations avec la variable cible
churn_corr = df[num_cols + [TARGET_CHURN]].corr()[TARGET_CHURN].drop(TARGET_CHURN).sort_values()

fig, ax = plt.subplots(figsize=(8, 7))
colors = ['#d62728' if v > 0 else '#2ca02c' for v in churn_corr.values]
churn_corr.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', lw=1)
ax.set_title('Corrélation des features avec le Churn', fontweight='bold')
ax.set_xlabel('Corrélation de Pearson')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_churn_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nTop 5 features positivement corrélées avec le churn :')
print(churn_corr.tail(5).to_string())
print('\nTop 5 features négativement corrélées avec le churn :')
print(churn_corr.head(5).to_string())

## 6. Analyse du Revenue at Risk

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Distribution du revenu total
df[TARGET_REVENUE].hist(ax=axes[0], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(df[TARGET_REVENUE].median(), color='red', lw=1.5, linestyle='--',
                label=f'Médiane : {df[TARGET_REVENUE].median():.0f}€')
axes[0].set_title('Distribution du revenu total (€)', fontweight='bold')
axes[0].set_xlabel('Revenu (€)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Revenu moyen : Churn vs Non-Churn
rev_by_churn = df.groupby(TARGET_CHURN)[TARGET_REVENUE].mean()
bars = axes[1].bar(['Non-Churn', 'Churn'], rev_by_churn.values,
                    color=['#2ca02c', '#d62728'], edgecolor='black')
for bar, val in zip(bars, rev_by_churn.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 5,
                 f'{val:.0f}€', ha='center', fontweight='bold')
axes[1].set_title('Revenu moyen — Churn vs Non-Churn', fontweight='bold')
axes[1].set_ylabel('Revenu moyen (€)')
axes[1].grid(axis='y', alpha=0.3)

# Revenue at Risk estimé
churn_clients = df[df[TARGET_CHURN] == 1]
rar = churn_clients[TARGET_REVENUE]
rar.hist(ax=axes[2], bins=40, color='coral', edgecolor='white')
axes[2].axvline(rar.median(), color='navy', lw=1.5, linestyle='--',
                label=f'Médiane : {rar.median():.0f}€')
axes[2].set_title('Revenu des clients churners (€)', fontweight='bold')
axes[2].set_xlabel('Revenu (€)')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle('Analyse du Revenue at Risk', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_revenue_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

total_rar = churn_clients[TARGET_REVENUE].sum()
total_rev = df[TARGET_REVENUE].sum()
print(f'Revenu total à risque : {total_rar:,.0f}€ ({total_rar/total_rev:.1%} du CA total)')

## 7. Détection des outliers (IQR)

In [ ]:
outlier_summary = []
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    outlier_summary.append({'feature': col, 'n_outliers': n_out,
                             'pct': round(n_out / len(df) * 100, 2)})

outlier_df = pd.DataFrame(outlier_summary).sort_values('pct', ascending=False)
print('Outliers détectés (méthode IQR) :')
display(outlier_df[outlier_df['n_outliers'] > 0])

# Boxplots des 6 features avec le plus d'outliers
top_out = outlier_df.head(6)['feature'].tolist()
if top_out:
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    for i, col in enumerate(top_out):
        df.boxplot(column=col, by=TARGET_CHURN, ax=axes[i],
                   boxprops={'color': 'steelblue'},
                   medianprops={'color': 'red', 'linewidth': 2})
        axes[i].set_title(col, fontweight='bold')
        axes[i].set_xlabel('')
    plt.suptitle('Boxplots — Top features avec outliers (par Churn)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'eda_boxplots_outliers.png', dpi=120, bbox_inches='tight')
    plt.show()

## 8. Synthèse et hypothèses pour la modélisation

In [ ]:
print('=' * 65)
print('SYNTHÈSE EDA — RÉSULTATS CLÉS')
print('=' * 65)
print(f'Dataset : {len(df):,} clients')
print(f'Taux de churn : {df[TARGET_CHURN].mean():.2%} → DÉSÉQUILIBRE FORT')
print(f'Features numériques : {len(num_cols)}')
print(f'Features catégorielles : {len([c for c in CATEGORICAL_FEATURES if c in df.columns])}')

missing = df.isnull().sum()
n_missing_cols = (missing > 0).sum()
print(f'Colonnes avec NaN : {n_missing_cols}')

print()
print('HYPOTHÈSES POUR LA MODÉLISATION :')
print('  1. Déséquilibre → utiliser SMOTE + class_weight + seuil abaissé')
print('  2. Métriques prioritaires : PR-AUC > ROC-AUC (classe rare)')
print('  3. Modèles non-linéaires (XGBoost, RF) probablement supérieurs')
print('  4. Features comportementales (logins, session_time) = signal fort')
print('  5. Features contrat (type, payment_failures) = signal fort')

print()
print('Figures EDA sauvegardées dans :', FIGURES_DIR)
print('=' * 65)